In [ ]:
# =====================================================================================
# セル1: 設定と、04番の出力 (05_graph_images_…) の検索
# =====================================================================================
# 06番: 04番が出力した各シートの「全データ総合比較」グラフを、データファイルどうしで並べ、
#        1つのPDFにまとめる (1ページ = 1シート)。ファイル名には作成年月、各ページには作成日時を入れる。
#        先に04番を、比べたいデータファイルすべてに実行しておくこと。ファイル選択ダイアログはない。

import os
import re
import glob
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image

# --- 場所の設定 (Jupyterで開くとカレントはコードフォルダになる) ---
PROJECT_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'コードフォルダ' else os.getcwd()
DATA_ROOT = os.path.join(PROJECT_DIR, 'データフォルダ', 'NR500')
OUTPUT_DIR = os.path.join(DATA_ROOT, '総合比較グラフ_データファイル間比較')

# --- 並べ方の設定 ---
MATERIAL_ORDER = ['pp', 'mixpp']              # 行 (上から)
MATERIAL_LABELS = {'pp': 'バージンPP（pp）', 'mixpp': '粉砕材混合PP（mixpp）'}
COUNT_ORDER = ['Alldata', '3data', '5data']   # 列 (左から)
KIND_LABELS = {'V01_plateau_end': 'V01横ばい終点版', 'max': '最大値版'}
# V01版と最大値版の両方にあるシート (先頭9列から作る5シート) は、この種類のグラフを使う。
# 先頭9列はV01版と最大値版でほぼ同じため (mixppは同じ。ppのV01版は1.5mm・3mm/sの3条件がない)
PREFERRED_KIND = 'max'
PAGE_SIZE = (11.69, 8.27)   # A4横 [inch]

# --- PDFの設定 ---
OUTPUT_PDF_BASENAME = '全データ総合比較_データファイル間比較'   # ファイル名は「{これ}_26年10月作成.pdf」のように作成年月を付ける
INCLUDE_THICKNESS_PAGES = False   # True にすると、各シートの全データ総合比較の前に、キャビ厚ごとの総合比較 (0.5〜2.0mm) のページも入れる

# --- 04番の出力を探す: <材料>_<種類>_<データ数> のフォルダにある 05_graph_images_… ---
FOLDER_PATTERN = re.compile(r'^(?P<mat>.+?)_(?P<kind>V01_plateau_end|max)_(?P<count>Alldata|3data|5data)$')
sources = {}   # (材料, 種類, データ数) -> 05_graph_images_… のパス
for graph_dir in sorted(glob.glob(os.path.join(DATA_ROOT, '*', '*', '*', '05_graph_images_*'))):
    if not os.path.isdir(graph_dir):
        continue
    m = FOLDER_PATTERN.match(os.path.basename(os.path.dirname(graph_dir)))
    if m is None:
        print(f"スキップ (フォルダ名が想定と違う): {graph_dir}")
        continue
    sources[(m['mat'], m['kind'], m['count'])] = graph_dir

# --- シートの一覧 (Excelと同じ並び: 両方の種類にあるシート → V01版だけのシート → 最大値版だけのシート。それぞれグラフ番号順) ---
sheet_kinds = {}     # シート名 -> そのシートがある種類
sheet_numbers = {}   # シート名 -> 最初のグラフ番号
for (mat, kind, count), graph_dir in sources.items():
    for sheet in os.listdir(graph_dir):
        sheet_dir = os.path.join(graph_dir, sheet)
        if not os.path.isdir(sheet_dir):
            continue
        sheet_kinds.setdefault(sheet, set()).add(kind)
        nums = [int(m.group(1)) for f in os.listdir(sheet_dir) if (m := re.match(r'^\[(\d+)\]_', f))]
        sheet_numbers[sheet] = min(nums + [sheet_numbers.get(sheet, float('inf'))])

def sheet_sort_key(sheet):
    kinds = sheet_kinds[sheet]
    group = 0 if len(kinds) > 1 else 1 + min(list(KIND_LABELS).index(k) if k in KIND_LABELS else 99 for k in kinds)
    return (group, sheet_numbers[sheet])

sheets = sorted(sheet_kinds, key=sheet_sort_key)
# シートごとに、どの種類のグラフを使うか
sheet_source_kind = {s: (PREFERRED_KIND if PREFERRED_KIND in sheet_kinds[s] else sorted(sheet_kinds[s])[0]) for s in sheets}

print(f"04番の出力: {len(sources)}ファイル分")
for (mat, kind, count), graph_dir in sources.items():
    print(f"  {mat:6s} {KIND_LABELS.get(kind, kind):10s} {count:8s} {os.path.relpath(graph_dir, DATA_ROOT)}")
print(f"\nPDFにするシート: {len(sheets)}個")
for i, sheet in enumerate(sheets, 1):
    print(f"  {i:02d}. {sheet}（{KIND_LABELS.get(sheet_source_kind[sheet], sheet_source_kind[sheet])}のグラフを使用）")


In [ ]:
# =====================================================================================
# セル2: PDFの作成 (全シートを1つのPDFにまとめる。1ページ = 1シートの全データ総合比較を、材料 × データ数で並べる)
# =====================================================================================

def set_japanese_font():
    standard_fonts = ['Yu Gothic', 'Meiryo', 'MS Gothic', 'Hiragino Sans', 'IPAexGothic']
    for font in standard_fonts:
        try:
            plt.rcParams['font.family'] = font; plt.rcParams['axes.unicode_minus'] = False; return
        except: continue
set_japanese_font()

# 総合比較グラフのPNG: キャビ厚ごと ([4]_CV0.5_sum_… / 両対数は [72]_CV0.5_log_sum_…) と、全データ ([17]_final_all_…)
THICKNESS_PATTERN = re.compile(r'^\[\d+\]_CV(?P<t>[\d.]+)_(?:log_)?sum_')
FINAL_PATTERN = re.compile(r'^\[\d+\]_final_all_')

def find_summary_pngs(sheet_dir):
    """シートのフォルダから、総合比較グラフのPNGを {ページのキー: パス} で返す (キー: キャビ厚の数値、全データは 'all')"""
    found = {}
    for f in sorted(os.listdir(sheet_dir)):
        if (m := THICKNESS_PATTERN.match(f)):
            found[float(m['t'])] = os.path.join(sheet_dir, f)
        elif FINAL_PATTERN.match(f):
            found['all'] = os.path.join(sheet_dir, f)
    return found

def page_label(key):
    return '全データ総合比較' if key == 'all' else f'キャビ厚 {key}mm の総合比較'

if not sources:
    print(f"エラー: 04番の出力 (05_graph_images_…) が {DATA_ROOT} に見つかりません。先に04番を実行してください。")
else:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    # 作成日時は1回だけ取り、ファイル名とすべてのページで同じものを使う
    # (ファイル名の月は2桁: 名前の順に並べると作成順になる。同じ月に作り直すと上書き、別の月なら別のファイルになる)
    created = datetime.now()
    pdf_name = f"{OUTPUT_PDF_BASENAME}_{created:%y}年{created:%m}月作成.pdf"
    pdf_path = os.path.join(OUTPUT_DIR, pdf_name)
    created_label = f"作成日時：{created:%Y年%m月%d日 %H:%M}"

    page_assignments = []   # 確認用: (ページ番号, シート名, ページの種類, 材料, データ数, PNGのパス)
    missing = []            # グラフがなかったマス
    page_no = 0
    with PdfPages(pdf_path, metadata={'Title': f'{OUTPUT_PDF_BASENAME}（{created:%Y-%m-%d %H:%M} 作成）', 'CreationDate': created}) as pdf:
        for sheet_no, sheet in enumerate(sheets, 1):
            kind = sheet_source_kind[sheet]
            pngs = {}   # (材料, データ数) -> {ページのキー: PNGのパス}
            for mat in MATERIAL_ORDER:
                for count in COUNT_ORDER:
                    graph_dir = sources.get((mat, kind, count))
                    sheet_dir = os.path.join(graph_dir, sheet) if graph_dir else None
                    pngs[(mat, count)] = find_summary_pngs(sheet_dir) if sheet_dir and os.path.isdir(sheet_dir) else {}
            thickness_keys = sorted({k for found in pngs.values() for k in found if k != 'all'})
            page_keys = (thickness_keys if INCLUDE_THICKNESS_PAGES else []) + ['all']

            for key in page_keys:
                page_no += 1
                fig = plt.figure(figsize=PAGE_SIZE)
                fig.suptitle(f"{sheet_no:02d}. {sheet}　{page_label(key)}", fontsize=14, fontweight='bold', y=0.975)
                fig.text(0.99, 0.985, created_label, ha='right', va='top', fontsize=9, color='dimgray')
                note = "データファイル間の比較（縦軸・横軸の範囲は全ファイル共通）"
                if len(sheet_kinds[sheet]) > 1:
                    note += f"。{KIND_LABELS.get(kind, kind)}のグラフを使用（V01横ばい終点版も同じ。ppのV01版は1.5mm・3mm/sの3条件なし）"
                fig.text(0.5, 0.93, note, ha='center', fontsize=9)
                grid = fig.add_gridspec(len(MATERIAL_ORDER), len(COUNT_ORDER), left=0.01, right=0.99, top=0.88, bottom=0.04, wspace=0.03, hspace=0.2)
                for r, mat in enumerate(MATERIAL_ORDER):
                    for c, count in enumerate(COUNT_ORDER):
                        ax = fig.add_subplot(grid[r, c])
                        ax.axis('off')
                        ax.set_title(f"{MATERIAL_LABELS.get(mat, mat)}・{KIND_LABELS.get(kind, kind)}・{count}", fontsize=9)
                        path = pngs[(mat, count)].get(key)
                        if path is None:
                            ax.text(0.5, 0.5, 'グラフなし', ha='center', va='center', fontsize=12, color='gray', transform=ax.transAxes)
                            missing.append((page_no, sheet, page_label(key), mat, count))
                            continue
                        # interpolation='none': PNGを縮小せず元の解像度でPDFに埋め込む (拡大しても読める)
                        ax.imshow(np.asarray(Image.open(path).convert('RGB')), interpolation='none')
                        ax.text(0.5, -0.01, os.path.basename(path), ha='center', va='top', fontsize=6, color='dimgray', transform=ax.transAxes)
                        page_assignments.append((page_no, sheet, page_label(key), mat, count, path))
                pdf.savefig(fig)
                plt.close(fig)

    print(f"保存: {pdf_path}")
    print(f"  {len(sheets)}シート・{page_no}ページ / {created_label}")
    print(f"  並べたグラフ: {len(page_assignments)}枚" + (f" / グラフがなかったマス: {len(missing)}個 {missing}" if missing else ""))
